In [ ]:

import pandas as pd
import numpy as np
import deeproot as dr

# 1) Carrega o CSV bruto vindo do node input
input = dr.input
df = pd.read_csv(input)

# 2) Padronizar strings (sem transformar NaN em 'nan')
for c in df.select_dtypes(include=["object"]):
    df[c] = df[c].astype("string").str.strip()

# 3) Remover IDs e colunas irrelevantes
drop_cols = [c for c in df.columns if c.lower() in ["customerid", "id"]]
df = df.drop(columns=drop_cols, errors="ignore")

# 4) Alvo binário (0/1)
if "Churn" not in df.columns:
    raise ValueError("Coluna 'Churn' não encontrada no dataset de entrada.")
df["Churn"] = df["Churn"].astype(str).str.lower().map({"yes":1, "no":0})

# 5) Converter TotalCharges para numérico (strings vazias -> NaN)
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 6) Imputação simples
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(df[c].median())
    else:
        df[c] = df[c].fillna("Unknown")

# 7) Remover colunas sem variação (1 valor único)
nun = df.nunique()
low_var = nun[nun <= 1].index.tolist()
df = df.drop(columns=low_var, errors="ignore")

# 8) Anti-leakage: remove qualquer coluna (exceto 'Churn') contendo 'churn' no nome
leaky = [c for c in df.columns if ("churn" in c.lower()) and (c != "Churn")]
df = df.drop(columns=leaky, errors="ignore")

# 8) salvar dataset preparado
dr.save_data(df, "telco_clean")
